# Llama 3.1 8B：Google Drive → Kaggle

使用 **CPU** 将固定版本的公开基础模型保存到 Google Drive，校验后上传为 `zhjianduan` 的私有 Kaggle Dataset。约 16.07 GB；不会加载模型或训练。

首次运行需要授权挂载 Drive。Kaggle 凭据只放在 Colab Secrets 或隐藏输入框中，不写入代码或 Drive。

来源：[NousResearch/Meta-Llama-3.1-8B](https://huggingface.co/NousResearch/Meta-Llama-3.1-8B)。Built with Llama；保留原许可证和使用政策。

## 1. 挂载 Google Drive

文件保存在下面的专用目录；下载中断后可以重跑，已校验的完整文件会跳过。

In [3]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
assert Path('/content/drive/MyDrive').is_dir(), 'Drive is not mounted'
MODEL_DIR = Path('/content/drive/MyDrive/Kaggle/name-that-llm/models/llama-3.1-8b-base-1f47e50')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print('Model directory:', MODEL_DIR)

Mounted at /content/drive
Model directory: /content/drive/MyDrive/Kaggle/name-that-llm/models/llama-3.1-8b-base-1f47e50


## 2. 固定文件清单与校验函数

四个权重分片使用官方仓库元数据中的 SHA-256 校验；其余文件使用固定版本的 Git blob hash 校验。仅下载 Transformers 所需文件，不重复下载 `original/`。

In [4]:
import json
MANIFEST = json.loads('{"repo": "NousResearch/Meta-Llama-3.1-8B", "revision": "1f47e50cdbe801ad8a5174156ec3a0655108fb9f", "official_revision": "d04e592bb4f6aa9cfee91e2e20afa771667e1d4b", "files": [{"name": "LICENSE", "size": 7627, "sha256": null, "git_blob_sha1": "a7c3ca16cee30425ed6ad841a809590f2bcbf290"}, {"name": "USE_POLICY.md", "size": 4691, "sha256": null, "git_blob_sha1": "81ebb55902285e8dd5804ccf423d17ffb2a622ee"}, {"name": "README.md", "size": 40896, "sha256": null, "git_blob_sha1": "b5a7a2779cc65e8458aecedd9b030a6f9afc7c01"}, {"name": "config.json", "size": 826, "sha256": null, "git_blob_sha1": "cccf055d6f8f210387a248c91dc40e0c7a4bafab"}, {"name": "generation_config.json", "size": 185, "sha256": null, "git_blob_sha1": "fc9506438b7b55383dc04c0816561442324846c3"}, {"name": "model.safetensors.index.json", "size": 23950, "sha256": null, "git_blob_sha1": "0fd8120f1c6acddc268ebc2583058efaf699a771"}, {"name": "special_tokens_map.json", "size": 73, "sha256": null, "git_blob_sha1": "d8cd5076496dbe4be2320312abc10adc43097b81"}, {"name": "tokenizer_config.json", "size": 50500, "sha256": null, "git_blob_sha1": "cb9ec25536e44d86778b10509d3e5bdca459a5cf"}, {"name": "tokenizer.json", "size": 9085658, "sha256": null, "git_blob_sha1": "f916e71031fa08f3c6ef1680a590c15b52d3cdd9"}, {"name": "model-00001-of-00004.safetensors", "size": 4976698672, "sha256": "f8b9704ab09cdeb097aa4a0a24bca96f906eec36bad63ab495bc21475058601b", "git_blob_sha1": "a70cf544ea82b645a9084d345203e3b855c474af"}, {"name": "model-00002-of-00004.safetensors", "size": 4999802720, "sha256": "c28b25e7541751056ee126627e007f8d4288319733285e9f7b17b9ff6eb313f0", "git_blob_sha1": "440f54a28d3e777128cca95168dd93017e99d8ef"}, {"name": "model-00003-of-00004.safetensors", "size": 4915916176, "sha256": "d8e9504dd4e4a146d484c52a97584ec14dac92237c46b064934af67a85e7d383", "git_blob_sha1": "b42588cf2677760fc6f6c39b19e55a31fca75e1c"}, {"name": "model-00004-of-00004.safetensors", "size": 1168138808, "sha256": "e4486f35c040f683f7d790354f66c169c109eb9fa0954a4a35d7c458a108405d", "git_blob_sha1": "e614ce32dfec70990b7029fb392a318df1622054"}]}')

In [5]:
"""Download a pinned public model into Drive, verifying every file before upload."""

import hashlib
import json
import os
import time
import urllib.error
import urllib.request
from pathlib import Path


def verify_file(path, entry):
    path = Path(path)
    if not path.is_file() or path.stat().st_size != entry['size']:
        return None
    sha256 = hashlib.sha256()
    git_blob = hashlib.sha1(f"blob {entry['size']}\0".encode())
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            sha256.update(chunk)
            git_blob.update(chunk)
    actual = sha256.hexdigest() if entry.get('sha256') else git_blob.hexdigest()
    expected = entry.get('sha256') or entry['git_blob_sha1']
    if actual != expected:
        return None
    return {'file': entry['name'], 'bytes': entry['size'], 'sha256': sha256.hexdigest()}


def download_verified(model_dir, manifest):
    model_dir = Path(model_dir)
    model_dir.mkdir(parents=True, exist_ok=True)
    verified = []
    started = time.monotonic()
    total = sum(item['size'] for item in manifest['files'])
    done = 0
    print(f"Downloading {len(manifest['files'])} files; {total / 1e9:.3f} GB total.", flush=True)
    for entry in manifest['files']:
        name = entry['name']
        if Path(name).name != name or name in ('.', '..'):
            raise ValueError('Only explicitly listed root-level model files are allowed.')
        dest = model_dir / name
        existing = verify_file(dest, entry)
        if existing:
            verified.append(existing)
            done += entry['size']
            print(f"Verified existing: {name}", flush=True)
            continue
        if dest.exists():
            raise RuntimeError(f"Existing file failed verification; left untouched: {dest}")
        partial = model_dir / (name + '.partial')
        url = f"https://huggingface.co/{manifest['repo']}/resolve/{manifest['revision']}/{name}"
        for attempt in range(1, 5):
            offset = partial.stat().st_size if partial.exists() else 0
            if offset > entry['size']:
                raise RuntimeError(f"Oversized partial file left untouched: {partial}")
            if offset == entry['size']:
                break
            headers = {'Accept-Encoding': 'identity', 'User-Agent': 'name-that-llm-model-transfer/1.0'}
            if offset:
                headers['Range'] = f'bytes={offset}-'
            last_report = time.monotonic()
            try:
                req = urllib.request.Request(url, headers=headers)
                with urllib.request.urlopen(req, timeout=180) as response:
                    if offset and response.status == 200:
                        offset = 0  # Server ignored Range: replace only our own incomplete file.
                    elif offset:
                        content_range = response.headers.get('Content-Range', '')
                        if response.status != 206 or not content_range.startswith(f'bytes {offset}-'):
                            raise RuntimeError(f'Unexpected resume response for {name}')
                    if response.status not in (200, 206):
                        raise RuntimeError(f'Unexpected HTTP status {response.status} for {name}')
                    with partial.open('ab' if offset else 'wb') as stream:
                        position = offset
                        while True:
                            chunk = response.read(8 * 1024 * 1024)
                            if not chunk:
                                break
                            if position + len(chunk) > entry['size']:
                                raise RuntimeError(f'Download exceeds expected size: {name}')
                            stream.write(chunk)
                            position += len(chunk)
                            now = time.monotonic()
                            if now - last_report >= 20:
                                print(f"{name}: {position / entry['size']:.1%}; total {(done + position) / total:.1%}", flush=True)
                                last_report = now
                if partial.stat().st_size == entry['size']:
                    break
                raise OSError('Connection ended before the expected file size.')
            except (OSError, urllib.error.URLError) as exc:
                # Do not print exception URLs: redirects can contain expiring signed parameters.
                if attempt == 4:
                    raise RuntimeError(f'Download interrupted for {name}; rerun to resume.') from None
                print(f"Retry {attempt}/4 for {name} ({type(exc).__name__}).", flush=True)
                time.sleep(min(5 * attempt, 15))
        print(f"Checking integrity: {name}", flush=True)
        result = verify_file(partial, entry)
        if result is None:
            raise RuntimeError(f'Integrity check failed; incomplete file preserved: {partial}')
        partial.replace(dest)
        verified.append(result)
        done += entry['size']
        print(f"Verified: {name}; {done / total:.1%} complete", flush=True)
    report = {
        'status': 'VERIFIED',
        'source': manifest,
        'files': verified,
        'bytes': total,
        'elapsed_seconds': round(time.monotonic() - started, 1),
    }
    (model_dir / 'transfer_manifest.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    print(f"All files verified. Persistent Drive directory: {model_dir}", flush=True)
    return report


def prepare_kaggle_dataset(model_dir, manifest):
    model_dir = Path(model_dir)
    report = json.loads((model_dir / 'transfer_manifest.json').read_text(encoding='utf-8'))
    assert report['status'] == 'VERIFIED'
    assert report['source'] == manifest
    expected = {item['name'] for item in manifest['files']}
    assert {item['file'] for item in report['files']} == expected
    for item in manifest['files']:
        assert (model_dir / item['name']).stat().st_size == item['size'], item['name']
    notice = 'Llama 3.1 is licensed under the Llama 3.1 Community License, Copyright © Meta Platforms, Inc. All Rights Reserved.\n'
    (model_dir / 'NOTICE').write_text(notice, encoding='utf-8')
    metadata = {
        'title': 'Llama 3.1 8B Base Verified Weights',
        'id': 'zhjianduan/llama31-8b-base-verified',
        'licenses': [{'name': 'other'}],
        'description': (
            'Built with Llama. Private transfer of the unmodified Llama 3.1 8B base Transformers weights. '
            'License: Llama 3.1 Community License; see LICENSE, USE_POLICY.md and NOTICE. '
            'Source: https://huggingface.co/' + manifest['repo'] + '/tree/' + manifest['revision'] + '. '
            'Weight SHA-256 metadata matches meta-llama/Llama-3.1-8B at ' + manifest['official_revision'] + '. '
            'Every downloaded file was verified before upload; see transfer_manifest.json. '
            'These are original base weights, not Instruct, GGUF, or competition adapters.'
        ),
    }
    (model_dir / 'dataset-metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    allowed = expected | {'transfer_manifest.json', 'NOTICE', 'dataset-metadata.json'}
    unexpected = [p.name for p in model_dir.iterdir() if p.name not in allowed]
    if unexpected:
        raise RuntimeError(f'Unexpected files must be reviewed before upload: {unexpected}')
    return metadata


## 3. 下载并逐文件校验

每约 20 秒显示下载进度。校验失败时停止，完整文件不会被覆盖。

In [6]:
verification = download_verified(MODEL_DIR, MANIFEST)
dataset_metadata = prepare_kaggle_dataset(MODEL_DIR, MANIFEST)
print({"status": verification["status"], "total_GB": round(verification["bytes"] / 1e9, 3), "dataset": dataset_metadata["id"]})

Checking integrity: LICENSE
Verified: LICENSE; 0.0% complete
Checking integrity: USE_POLICY.md
Verified: USE_POLICY.md; 0.0% complete
Checking integrity: README.md
Verified: README.md; 0.0% complete
Checking integrity: config.json
Verified: config.json; 0.0% complete
Checking integrity: generation_config.json
Verified: generation_config.json; 0.0% complete
Checking integrity: model.safetensors.index.json
Verified: model.safetensors.index.json; 0.0% complete
Checking integrity: special_tokens_map.json
Verified: special_tokens_map.json; 0.0% complete
Checking integrity: tokenizer_config.json
Verified: tokenizer_config.json; 0.0% complete
Checking integrity: tokenizer.json
Verified: tokenizer.json; 0.1% complete
model-00001-of-00004.safetensors: 38.9%; total 12.1%
model-00001-of-00004.safetensors: 66.4%; total 20.6%
model-00001-of-00004.safetensors: 98.4%; total 30.5%
Checking integrity: model-00001-of-00004.safetensors
Verified: model-00001-of-00004.safetensors; 31.0% complete
model-0000

## 4. 登录 Kaggle

优先读取 Colab Secrets 中名为 `KAGGLE_API_TOKEN` 的密钥；没有时会出现隐藏输入框。可在 [Kaggle 设置](https://www.kaggle.com/settings) 的 API 区域生成 token。不要把 token 粘贴到代码或聊天中。

In [7]:
import os, sys, subprocess, getpass
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle==2.2.5'], check=True)
from google.colab import userdata
try:
    kaggle_token = userdata.get('KAGGLE_API_TOKEN')
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    kaggle_token = getpass.getpass('Kaggle API token (hidden): ')
if not kaggle_token or not kaggle_token.strip():
    raise RuntimeError('Kaggle token is required for upload; downloaded files are safe in Drive.')
os.environ['KAGGLE_API_TOKEN'] = kaggle_token.strip()
del kaggle_token
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()
username = api.config_values.get('username')
assert username == 'zhjianduan', f'Wrong Kaggle account: {username}'
print('Authenticated Kaggle account:', username)

Authenticated Kaggle account: zhjianduan


## 5. 上传私有 Dataset

仅上传已校验的模型目录。保留 Llama 3.1 许可，默认私有；若同名 Dataset 已存在，停止创建并保留原版本。

In [9]:
"""Create once, preserve a receipt, and report a status-read failure separately."""
from requests.exceptions import HTTPError

dataset_metadata = prepare_kaggle_dataset(MODEL_DIR, MANIFEST)
dataset_ref = dataset_metadata['id']
receipt_path = MODEL_DIR.parent / (MODEL_DIR.name + '-kaggle-receipt.json')
if receipt_path.exists():
    receipt = json.loads(receipt_path.read_text(encoding='utf-8'))
    assert receipt['dataset_ref'] == dataset_ref
    assert receipt['source_revision'] == MANIFEST['revision']
    assert receipt['creation_accepted'] is True
    print('Using the existing upload receipt:', receipt['dataset_url'])
else:
    # Reuse the accepted result when only the following status lookup failed.
    if 'result' not in globals() or not getattr(result, 'url', None):
        result = api.dataset_create_new(
            str(MODEL_DIR), public=False, quiet=False,
            convert_to_csv=False, dir_mode='skip',
        )
    if getattr(result, 'error', None):
        raise RuntimeError(result.error)
    expected_url = 'https://www.kaggle.com/datasets/' + dataset_ref
    assert getattr(result, 'url', None) == expected_url
    receipt = {
        'dataset_ref': dataset_ref,
        'dataset_url': result.url,
        'source_revision': MANIFEST['revision'],
        'creation_accepted': True,
        'private_requested': True,
        'expected_files': len(MANIFEST['files']) + 2,
    }
    receipt_path.write_text(json.dumps(receipt, indent=2), encoding='utf-8')
    print('Dataset creation accepted:', receipt['dataset_url'])

try:
    status = api.dataset_status(dataset_ref)
    print('Kaggle processing status:', status)
    receipt['api_status_read'] = 'SUCCESS'
    receipt['api_status'] = status if isinstance(status, (dict, str, list)) else str(status)
except HTTPError as exc:
    if exc.response is None or exc.response.status_code != 403:
        raise
    receipt['api_status_read'] = '403_FORBIDDEN'
    print('Upload was accepted, but the API status lookup returned 403.')
    print('Check the private dataset page for version, file count and readiness:', receipt['dataset_url'])
finally:
    _ = os.environ.pop('KAGGLE_API_TOKEN', None)
    receipt_path.write_text(json.dumps(receipt, indent=2), encoding='utf-8')
print('Upload receipt saved in Drive:', receipt_path)


Dataset creation accepted: https://www.kaggle.com/datasets/zhjianduan/llama31-8b-base-verified
Kaggle processing status: ready
Upload receipt saved in Drive: /content/drive/MyDrive/Kaggle/name-that-llm/models/llama-3.1-8b-base-1f47e50-kaggle-receipt.json


## 6. 完成后

确认 Kaggle Dataset 处理完成、文件齐全后，将它挂载到比赛 Notebook 并修改模型路径。这一步完成的是权重传输；比赛推理、分数验证需要单独运行。